# Dense Evolution — Live Dashboard on Colab

This notebook runs the real Dense Evolution Streamlit dashboard on Google Colab's free compute and gives you a clickable link to open it, just like a hosted app.

**How to use:** Run the two cells below in order, then click the `https://xxxx.trycloudflare.com` link printed at the end of the second cell's output. No password or account needed.

Project: [github.com/tatopenn-cell/Dense-Evolution](https://github.com/tatopenn-cell/Dense-Evolution)

In [ ]:
# 1. Clone the repo and install dependencies (CPU-only extras, fast on Colab's free tier)
!git clone -q https://github.com/tatopenn-cell/Dense-Evolution.git
%cd Dense-Evolution
!pip install -q -e ".[dashboard,jax]"

In [ ]:
# 2. Launch the dashboard and expose it with a public tunnel (Cloudflare Tunnel — no account needed)
import time, re

get_ipython().system_raw(
    "streamlit run app_dashboard.py --server.port 8501 "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.headless true &>/content/logs.txt &"
)

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

get_ipython().system_raw("./cloudflared tunnel --url http://localhost:8501 &>/content/cloudflared.log &")

print("Starting dashboard + tunnel, this takes ~15-20s...")
url = None
for _ in range(30):
    time.sleep(1)
    try:
        log = open("/content/cloudflared.log").read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
        if match:
            url = match.group(0)
            break
    except FileNotFoundError:
        pass

print(f"\nOpen the dashboard here: {url}" if url else "\nTunnel URL not found yet — check /content/cloudflared.log")

---
If the link doesn't load right away, wait a few seconds and refresh — Streamlit can take a moment to finish starting up. To stop the app, interrupt/stop the second cell.